## Временная миграция VS Глубинная
В этом ноутбуке на примере миграции Кирхгофа разберем отличие временной и глубинной миграции.
Чтобы упростить задачу и не писать килотонны кода, воспользуемся уже знакомым нам оператором из библиотеки PyLops. 
И будем делать не строго прям временную миграцию, а её фактический аналог в глубине (так как наш чудесный оператор не расчитан на работу во временах), который может быть пересчитан во времена простой интерполяцией.

Напомню основной тезис:

* Временная миграция исходит из предположения о горизонтально-слоистой модели и берет локально-постоянную скорость в каждой точке среды, и соответственно времена пробега волны берет по прямому лучу. Скорость для такой миграции обычно задается через Vrms (t) - но в нашем варианте это будет Vrms (z).

**Ключевая формула для Vrms:**

$$V_{rms}(z) = \sqrt{\frac{\int_0^z v^2(\zeta) \, d\zeta}{\int_0^z \frac{d\zeta}{v(\zeta)}}}$$

где $v(\zeta)$ - интервальная скорость на глубине $\zeta$.

Чтобы сразу показать наглядно и ограничения временной миграции, возмём модельку с чем-то вроде соляного купола (весьма схематичного). 

План работы:

* Создание модели и съемки
* Моделирование сейсмограмм  Kircchoff forward оператором.
* Получение глубинной миграции через adjoint оператор
* Создание Vrms и времен пробега для временной миграции
* Получение псевдо-временной миграции в через adjoint-оператор с кастомными временами пробега
* Сравнение результатов



In [ ]:
# !pip install segyio matplotlib tqdm scipy pylops numba
# numba нужна для ускорения вычислений в pylops 

import segyio
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from pylops.waveeqprocessing import Kirchhoff
import warnings
warnings.filterwarnings('ignore')

## 1. Чтение модели из SEG-Y, создание модели коэффициента отражения и съемки

In [ ]:
MODEL_FILE = 'vmodel_dome.sgy'

# 1. Чтение модели. Воспользуемся библиотекой segyio
with segyio.open(MODEL_FILE, ignore_geometry=True) as f:
    # В segyio данные хранятся как (ntraces, nsamples)
    # .T дает (nsamples, ntraces) = (глубина, x)
    v = f.trace.raw[:].T
    x = f.attributes(segyio.TraceField.SourceX)[:]
    z = f.samples  # глубины в метрах

dx = x[1] - x[0]
dz = z[1] - z[0]
nz, nx = v.shape

print(f'Исходная модель: {nz} отсчетов по глубине, {nx} трасс')
print(f'dx = {dx:.2f} м, dz = {dz:.2f} м')


# 2. Создание модели коэффициента отражения (reflectivity) из модели скоростей, принимая плотность постоянной

# Вычисляем коэффициент отражения по вертикали (изменения по глубине)
reflectivity = np.zeros_like(v)
reflectivity[1:, :] = (v[1:, :] - v[:-1, :]) / (v[1:, :] + v[:-1, :])

# Добавим дифракторы
reflectivity[200:202, 150:152] += 0.3
reflectivity[250:252, 300:302] += 0.2

print(f'Reflectivity модель создана')
print(f'Min reflectivity: {reflectivity.min():.4f}, Max: {reflectivity.max():.4f}')

# Параметры источника
sx_idx =  np.arange(0, nx + 1, 50, dtype=int)
sz_idx = np.zeros(len(sx_idx), dtype=int) 
sx = x[sx_idx]
sz = z[sz_idx]

# Параметры приемников - через каждые 10 точек сетки (50 м)
rx_idx = np.arange(0, nx + 1, 10, dtype=int)
rz_idx = np.zeros(len(rx_idx), dtype=int) 
rx = x[rx_idx]
rz = z[rz_idx]

# 3. Визуализация модели скоростей и reflectivity рядом
fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# Модель скоростей
im1 = axes[0].imshow(v, aspect='equal', cmap='jet',
                      extent=[x[0], x[-1], z[-1], z[0]])    
axes[0].scatter(sx, sz, c='yellow', marker='v', s=50, label='Источники')
axes[0].scatter(rx, rz, c='cyan', marker='o', s=25, label='Приемники', alpha=0.5)
axes[0].legend()
axes[0].set_ylabel('Z, (м)')
axes[0].set_title('Скоростная модель')
cbar1 = plt.colorbar(im1, ax=axes[0], label='Скорость (м/с)')

# Reflectivity модель
im2 = axes[1].imshow(reflectivity, aspect='equal', cmap='RdBu_r',
                      extent=[x[0], x[-1], z[-1], z[0]],)
axes[1].scatter(sx, sz, c='red', marker='v', s=50, label='Источники')
axes[1].scatter(rx, rz, c='blue', marker='o', s=25, label='Приемники', alpha=0.5)
axes[1].legend()
axes[1].set_ylabel('Z, (м)')
axes[1].set_xlabel('X (м)')
axes[1].set_title('Модель Reflectivity')
cbar2 = plt.colorbar(im2, ax=axes[1], label='Коэффициент отражения')

plt.tight_layout()

### Задание параметров сейсмического эксперимента: время и импульс

In [ ]:
from pylops.utils.wavelets import ricker
# Временная ось (сек)
dt = 0.002  # 2 мс
tmax = 2.5  # 2.5 с, достаточно для двукратного пробега до 2 км
nt = int(tmax / dt) + 1
t = np.arange(nt) * dt

# Риккеровский вэйвлет
f0 = 20.0  # Гц
wav, tw, wawc = ricker(t[:41], f0)  

## 2. Прямое моделирование для получения сейсмограмм

In [ ]:
srcs = np.vstack([sx, sz]).astype(float)
recs = np.vstack([rx, rz]).astype(float)           
nsrcs = srcs.shape[1]
nrecs = recs.shape[1]

print(f"nsrcs={nsrcs}, nrecs={nrecs}, nt={nt}")

# Создаем оператор Кирхгофа
K = Kirchhoff(
    z=z, x=x, t=t,
    srcs=srcs, recs=recs,
    vel=v.T,
    wav=wav, wavcenter=wawc,
    mode='eikonal', # расчет времен пробега по эйконалу с помощью scikit-fmm методом fast marching
    angleaperture=60.0,
    dynamic=True, # для применения амплитудных коррекций за расхождение фронта волны
    engine='numba', # с numba гораздо быстрее, можно поставить numpy
    dtype='float64', 
    name='K')

In [ ]:
# Для применения оператора Кирхгофа нужно векторизовать модель (превратить в одномерный массив). 
# Порядок "F" означает, что массив будет считан по столбцам.
mvec = reflectivity.ravel(order="F")
dvec = K @ mvec  # размер (nt * nrecs * nsrcs,)

# Преобразуем в (nt, nrecs, nsrcs)
seismograms = dvec.reshape(nt, nrecs, nsrcs, order='F')

In [ ]:
# Визуализация сейсмограмм для нескольких источников
fig, axes = plt.subplots(1, 3, figsize=(10, 5), sharey=True)
nsrc_to_plot = (1, 5, 9)
for src_index in nsrc_to_plot:
    ax = axes[nsrc_to_plot.index(src_index)]
    im = ax.imshow(seismograms[:, :, src_index], aspect='auto', cmap='gray',
                   extent=[rx[0], rx[-1], t[-1], t[0]])
    ax.set_title(f'Source {src_index+1} (x={sx[src_index]:.1f} м)')
    ax.set_xlabel('X, m')
    ax.set_ylabel('T, s')
fig.tight_layout()

## 3. Получение глубинной миграции
Применим тот же оператор в adjoint-режиме

Глубинная миграция использует точную трассировку лучей (уравнение эйконала) и интервальные скорости $v(z,x)$:

$$\left|\nabla t\right| = \frac{1}{v(z,x)}$$

где $t$ - время пробега, вычисляемое методом fast marching.

In [ ]:
depth_migration = K.H @ dvec
depth_migration = depth_migration.reshape(nz, nx, order='F')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
axs[0].imshow(reflectivity, aspect='equal', cmap='gray', extent=[x[0], x[-1], z[-1], z[0]])
axs[0].set_title('Исходная модель отражаемости')
axs[0].set_xlabel('X, m')
axs[0].set_ylabel('Z, m')
vmin, vmax = np.percentile(depth_migration, [1, 99])
axs[1].imshow(depth_migration, aspect='equal', cmap='gray', extent=[x[0], x[-1], z[-1], z[0]], vmin=vmin, vmax=vmax)
axs[1].set_title('Глубинная миграция (эйконал)')
axs[1].set_xlabel('X, m')
fig.tight_layout()

## 4. Подготовка массивов для временной миграции

Для временной миграции вычисляем Vrms по формуле:

$$V_{rms}^2(z) = \frac{\sum_{i=0}^{z} v_i^2 \Delta z_i}{\sum_{i=0}^{z} \frac{\Delta z_i}{v_i}}$$ 

In [ ]:
from scipy.ndimage import gaussian_filter

dz_layers = np.gradient(z) 

# Итоговый массив Vrms(nz, nx)
Vrms = np.zeros_like(v)

for ix in range(nx):
    v_col = v[:, ix]
    
    # 1. Рассчитываем интервальные времена пробега (одноходовые)
    # Добавляем небольшое число (эпсилон) в знаменатель для стабильности, если v_col[0] = 0
    dt_col = dz_layers / (v_col + 1e-10) 
    
    # 2. Рассчитываем числитель для формулы Vrms^2 (интеграл от v^2 * dt)
    # v^2 * dt = v^2 * (dz / v) = v * dz
    numerator = np.cumsum(v_col * dz_layers)
    
    # 3. Рассчитываем знаменатель (общее одноходовое время)
    denominator = np.cumsum(dt_col)
    
    # --- Обработка деления на ноль для первой точки ---
    # В первой точке (z=0) время равно 0, что вызовет ошибку.
    # Vrms на поверхности равна интервальной скорости на поверхности.
    Vrms[0, ix] = v_col[0]
    
    # Рассчитываем Vrms для всех остальных точек, где время > 0
    # Используем np.divide с условием where, чтобы избежать ошибок
    non_zero_time_mask = denominator > 1e-10
    
    # Вычисляем квадрат Vrms
    vrms_sq = np.divide(numerator, denominator, where=non_zero_time_mask)
    
    # Берем корень и присваиваем значения, избегая отрицательных подкоренных выражений
    Vrms[non_zero_time_mask, ix] = np.sqrt(np.maximum(0, vrms_sq[non_zero_time_mask]))


# сгладим Vrms 

Vrms = gaussian_filter(Vrms, sigma=10.0)    

fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
axs[0].imshow(v, aspect='equal', cmap='jet', 
               extent=[x[0], x[-1], z[-1], z[0]])
axs[0].set_title('Исходная модель v(z,x)')
axs[0].set_xlabel('X, m')
axs[0].set_ylabel('Z, m')

axs[1].imshow(Vrms, aspect='equal', cmap='jet',
               extent=[0, x[-1],  z[-1], z[0]])
axs[1].set_title('Vrms(z,x)')
axs[1].set_xlabel('X, m')
plt.tight_layout()

Оператор Кирхгофа использует функцию Грина с амплитудными коэффициентами и временами пробега:

$$G(\mathbf{x}_r, \mathbf{x}_s, \mathbf{x}, t) = A_{src}(\mathbf{x}) \cdot A_{rec}(\mathbf{x}) \cdot \delta(t - \tau_{src}(\mathbf{x}) - \tau_{rec}(\mathbf{x}))$$

где:
- $A_{src}(\mathbf{x})$, $A_{rec}(\mathbf{x})$ - амплитудные коэффициенты от источника и до приемника
- $\tau_{src}(\mathbf{x})$, $\tau_{rec}(\mathbf{x})$ - времена пробега от источника и до приемника
- $\delta$ - дельта-функция Дирака

Так как в Pylops не предусмотрен расчет времен пробега и амплитудных коэффициентов для временной миграции, реализуем кастомный вариант

Вычисление времен пробега через Vrms и амплитудных коэффициентов

Для временной миграции используем прямые лучи (горизонтально-слоистая модель):

**Время пробега:**
$$t = \frac{\sqrt{\Delta x^2 + z^2}}{V_{rms}(z)}$$

**Амплитудный коэффициент** (геометрическое расхождение):
$$A = \frac{1}{\sqrt{\Delta x^2 + z^2}}$$ 

In [ ]:
# Инициализация массивов времен пробега
traveltimes_src = np.zeros((nx * nz, nsrcs), dtype=np.float32)
traveltimes_rec = np.zeros((nx * nz, nrecs), dtype=np.float32)

# Инициализация массивов амплитудных коэффициентов

amplitudes_src = np.zeros((nx * nz, nsrcs), dtype=np.float32)
amplitudes_rec = np.zeros((nx * nz, nrecs), dtype=np.float32)
epsilon = 1e-9

# Основной цикл по точкам сетки миграции
for iz in tqdm(range(nz)):
    for ix, x_ in enumerate(x):       
        # Получаем локальнуюскорость для текущей точки (iz, ix)
        v_ = Vrms[iz, ix]         
      
        image_point_idx = ix * nz + iz

        # Горизонтальные расстояния от текущей точки до всех источников/приемников
        dx_srcs = x_ - srcs[0]
        dx_recs = x_ - recs[0]

        # 1. Рассчитываем полные расстояния
        dist_srcs = np.sqrt(dx_srcs**2 + z[iz]**2)
        dist_recs = np.sqrt(dx_recs**2 + z[iz]**2)

        # 2. Рассчитываем времена, используя уже вычисленные расстояния
        traveltimes_src[image_point_idx, :] = dist_srcs / v_
        traveltimes_rec[image_point_idx, :] = dist_recs / v_

        # 3. Рассчитываем амплитуды по формуле a = 1 / sqrt(dist)
        #    Используем epsilon для избежания ошибки 1/sqrt(0)
        amplitudes_src[image_point_idx, :] = 1.0 / np.sqrt(dist_srcs + epsilon)
        amplitudes_rec[image_point_idx, :] = 1.0 / np.sqrt(dist_recs + epsilon) 

## 5. Временная миграция
Собственно, наша псевдо-временная миграция в глубине


In [ ]:
KTMigOp = Kirchhoff(
    z=z, x=x, t=t,
    srcs=srcs, recs=recs,
    vel=1.0, # фиктивная скорость, т.к. мы используем свои времена пробега
    trav=(traveltimes_src, traveltimes_rec), # наши времена пробега
    amp=(amplitudes_src, amplitudes_rec), # наши амплитудные коэффициенты
    wav=wav, wavcenter=wawc,
    mode='byot', # "bring your own traveltimes" - кастомные времена пробега.     
    angleaperture=60.0,
    dynamic=True, # для применения амплитудных коррекций 
    engine='numba',
    name='KTmigOp'
)

In [ ]:
time_migration = KTMigOp.H @ dvec
time_migration = time_migration.reshape(nz, nx, order='F')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
axs[0].imshow(reflectivity, aspect='equal', cmap='gray', extent=[x[0], x[-1], z[-1], z[0]])
axs[0].set_title('Исходная модель отражаемости')
vmin, vmax = np.percentile(time_migration, [1, 99])
axs[1].imshow(time_migration, aspect='equal', cmap='gray', extent=[x[0], x[-1], z[-1], z[0]], vmin=vmin, vmax=vmax)
axs[1].set_title('Временная миграция в глубине')
fig.tight_layout()

## 6. Сравнение результатов

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
vmin, vmax = np.percentile(depth_migration, [1, 99])
axs[0].imshow(depth_migration, aspect='equal', cmap='gray', extent=[x[0], x[-1], z[-1], z[0]], vmin=vmin, vmax=vmax)
axs[0].set_title('Глубинная миграция (Vint + eikonal)')
axs[0].set_xlabel('X, m')
axs[0].set_ylabel('Z, m')
vmin, vmax = np.percentile(time_migration, [1, 99])
axs[1].imshow(time_migration, aspect='equal', cmap='gray', extent=[x[0], x[-1], z[-1], z[0]], vmin=vmin, vmax=vmax)
axs[1].set_title('"Временная миграция" (по прямому лучу с Vrms)')
axs[1].set_xlabel('X, m')
fig.tight_layout()

На изображениях хорошо видно, что около-горизонтальные участки и дифрактор вне купола отлично мигрируются временной миграцией, но вот сама граница купола и всё что под ней - уже извините. Тут требуется точная трассировка лучей, а временная миграция работает с горизонтально-слоистой моделью и прямыми лучами, что в случае вот таких куполов абсолютно не адекватно

Кстати, и на глубинной купол прорисовался неидеально - тут уже мы подошли к границе возможностей лучевого приближения, нужно полноволновое моделирование